# Modeling Evaluation
All modeling logic is now housed in `src/modeling.py`. This notebook serves as a wrapper to view the pipeline results and ensures consistency with the main script.

In [3]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.data_loader import load_data
from src.features import engineer_features, get_feature_columns
from src.modeling import train_and_evaluate_oof

# Load and process data using the central src modules
df = load_data()
df = engineer_features(df)
num_cols, cat_cols = get_feature_columns()

# Run OOF validation
metrics, oof_proba, y_true = train_and_evaluate_oof(df, num_cols, cat_cols, 'Customer_ID')

print("\n--- Final Metrics ---")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

Running 5-Fold Stratified Group CV on 100000 rows...

--- Final Metrics ---
Base_Rate: 0.2399
Dummy_ROC_AUC_Mean: 0.4986
Dummy_ROC_AUC_Std: 0.0044
Dummy_Avg_Prec_Mean: 0.2394
LR_ROC_AUC_Mean: 0.5000
LR_ROC_AUC_Std: 0.0035
LR_Avg_Prec_Mean: 0.2416
RF_ROC_AUC_Mean: 0.5021
RF_ROC_AUC_Std: 0.0068
RF_Avg_Prec_Mean: 0.2422


In [4]:
import os

test_path = r"C:\Users\boses\Desktop\bfsi-fraud-project\tests\test_data.py"

content = """import sys
import pytest
import pandas as pd
from pathlib import Path

sys.path.append(str(Path(__file__).resolve().parent.parent))
from src.validation import validate_transactions

@pytest.fixture
def sample_df():
    return pd.DataFrame({
        'Transaction_ID': ['T1', 'T2', 'T3'],
        'Date': ['12/02/22', '13/02/22', '14/02/22'],
        'Time': ['08:20:00 AM', '09:00:00 AM', '10:00:00 AM'],
        'Merchant_ID': ['M1', 'M1', 'M2'],
        'Customer_ID': ['C1', 'C1', 'C2'],
        'Device_ID': ['D1', 'D1', 'D2'],
        'Transaction_Type': ['Purchase', 'Purchase', 'Refund'],
        'Payment_Gateway': ['A', 'A', 'B'],
        'Transaction_City': ['X', 'X', 'Y'],
        'Transaction_State': ['S1', 'S1', 'S2'],
        'IP_Address': ['1.1.1.1', '1.1.1.2', '2.2.2.2'],
        'Transaction_Status': ['Completed', 'Completed', 'Pending'],
        'Device_OS': ['Android', 'Android', 'iOS'],
        'Transaction_Frequency': [0, 1, 0],
        'Merchant_Category': ['Retail', 'Retail', 'Services'],
        'Transaction_Channel': ['Mobile', 'Mobile', 'Online'],
        'Transaction_Amount_Deviation': [10.0, 15.0, -5.0],
        'Days_Since_Last_Transaction': [10, 1, 5],
        'amount': [100.0, 200.0, 150.0],
        'fraud': [0, 1, 0]
    })

def test_validation_passes_on_valid_data(sample_df):
    assert validate_transactions(sample_df) is True

def test_validation_catches_missing_columns(sample_df):
    broken_df = sample_df.drop(columns=['amount'])
    with pytest.raises(ValueError, match="Missing required columns"):
        validate_transactions(broken_df)

def test_validation_catches_invalid_fraud_labels(sample_df):
    sample_df.loc[0, 'fraud'] = 2
    with pytest.raises(ValueError, match="Invalid 'fraud' labels found"):
        validate_transactions(sample_df)
"""

with open(test_path, "w", encoding="utf-8") as f:
    f.write(content)

print("tests/test_data.py successfully overwritten with mock data.")

tests/test_data.py successfully overwritten with mock data.


In [5]:
import os
from fpdf import FPDF

class PDF(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 16)
        self.cell(0, 10, 'UPI Fraud Detection: Project Summary', 0, 1, 'C')
        self.ln(5)

def create_memo():
    pdf = PDF()
    pdf.add_page()
    
    pdf.set_font('Helvetica', 'B', 11)
    pdf.cell(0, 6, 'TO: Fraud Risk Team Lead', 0, 1)
    pdf.cell(0, 6, 'SUBJECT: Findings from Synthetic UPI Transaction Analysis', 0, 1)
    pdf.ln(5)

    pdf.set_font('Helvetica', 'B', 12)
    pdf.cell(0, 8, '1. The Business Question', 0, 1)
    pdf.set_font('Helvetica', '', 11)
    pdf.multi_cell(0, 6, "Which transaction patterns in a UPI payment flow are most predictive of fraud, and what is the precision-recall trade-off at different risk thresholds?")
    pdf.ln(3)

    pdf.set_font('Helvetica', 'B', 12)
    pdf.cell(0, 8, '2. Dataset Context & Limitations', 0, 1)
    pdf.set_font('Helvetica', '', 11)
    pdf.multi_cell(0, 6, "The analysis was conducted on a 100,000-row synthetic UPI dataset. The fraud rate is 23.99%. A critical limitation was identified: the dataset includes pre-computed historical features (e.g., Transaction_Frequency) that may contain future information leakage. Furthermore, we removed Transaction_Status because it is a post-transaction metric unavailable at real-time scoring.")
    pdf.ln(3)

    pdf.set_font('Helvetica', 'B', 12)
    pdf.cell(0, 8, '3. Analytical Investigations', 0, 1)
    pdf.set_font('Helvetica', '', 11)
    pdf.multi_cell(0, 6, "1. SQL Layer: We utilized window functions to analyze transaction velocity. Results showed a flat fraud rate across all time-gap buckets, indicating no velocity signal.\n2. Machine Learning: We trained Logistic Regression and Random Forest models using StratifiedGroupKFold by Customer_ID. This prevents the model from memorizing specific customers.")
    pdf.ln(3)

    pdf.set_font('Helvetica', 'B', 12)
    pdf.cell(0, 8, '4. Findings & Null Results', 0, 1)
    pdf.set_font('Helvetica', '', 11)
    pdf.multi_cell(0, 6, "Both models achieved an Out-of-Fold ROC-AUC of ~0.50 (equivalent to random chance). The baseline Dummy Classifier scored 0.4986, proving the models learned nothing.\n\nBusiness Cost Simulation (using OOF predictions):\n- At Threshold 0.3: The model flags all 100,000 transactions. Investigation cost = Rs 2,000,000.\n- At Threshold 0.7: The model misses all 23,986 frauds. Missed fraud cost = Rs 11,993,000.\nThe model fails to provide a cost-effective fraud filter.")
    pdf.ln(3)

    pdf.set_font('Helvetica', 'B', 12)
    pdf.cell(0, 8, '5. Business Implications', 0, 1)
    pdf.set_font('Helvetica', '', 11)
    pdf.multi_cell(0, 6, "Because the model has no predictive power, deploying it would result in massive false-positive investigation costs or huge missed-fraud losses. The data science pipeline is robust and leakage-free, but the synthetic dataset lacks the complex network and behavioral signals of real UPI traffic.")
    pdf.ln(5)

    pdf.set_font('Helvetica', 'I', 10)
    pdf.cell(0, 6, "Code and reproducibility instructions: github.com/bosesrinjoy24/upi-fraud-risk-project", 0, 1, 'C')

    # Save to the memo folder (one level up from the notebooks folder)
    out_dir = os.path.join("..", "memo")
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, "UPI_Fraud_Project_Summary.pdf")
    pdf.output(out_path)
    print(f"Final Memo generated at: {out_path}")

create_memo()

Final Memo generated at: ..\memo\UPI_Fraud_Project_Summary.pdf
